# Phase 1 — Ingestion Validation

Proves the five streams landed in Postgres and that they **join** — the whole point of the archaeologist.

**Run first** (from the repo root):
```
uv run python -m archaeologist.ingestion.run --skip-issues   # code + docs + git
uv run python -m archaeologist.ingestion.run --only-issues   # issues/PRs (add GITHUB_TOKEN for the full pull)
```
Then run all cells here.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from sqlalchemy import func, select, text
from archaeologist.models.db import session_scope
from archaeologist.models.entities import Commit, CommitFile, File, Issue, Repo
print("repo root:", root)

## Row counts per stream

In [ ]:
with session_scope() as s:
    for name, model in [("repos", Repo), ("files", File), ("commits", Commit),
                        ("commit_files", CommitFile), ("issues/PRs", Issue)]:
        print(f"  {name:13}: {s.scalar(select(func.count()).select_from(model))}")
    repo = s.scalar(select(Repo))
    print(f"\n  target: {repo.name}  branch={repo.default_branch}  head={(repo.head_sha or '')[:8]}")

## Files by stream category (code / doc / config / test / other)

In [ ]:
with session_scope() as s:
    for cat, n in s.execute(select(File.category, func.count())
                            .group_by(File.category).order_by(func.count().desc())):
        print(f"  {cat:8}: {'#' * n}  {n}")

## Cross-stream join #1 — most-changed files
Joins the **git-history** stream to the **code** stream by `path`. High churn = a hotspot worth investigating.

In [ ]:
with session_scope() as s:
    rows = s.execute(select(CommitFile.path, func.count().label("changes"))
                     .group_by(CommitFile.path).order_by(text("changes DESC")).limit(10))
    for path, changes in rows:
        print(f"  {changes:4}  {path}")

## Cross-stream join #2 — the history of one file
"Why did this file change?" — every commit that touched `src/flask/app.py`, newest first. This is the archaeologist's core move.

In [ ]:
TARGET = "src/flask/app.py"
with session_scope() as s:
    rows = s.execute(
        select(Commit.sha, Commit.authored_at, Commit.author_name, Commit.message)
        .join(CommitFile, (CommitFile.commit_sha == Commit.sha) & (CommitFile.repo_id == Commit.repo_id))
        .where(CommitFile.path == TARGET)
        .order_by(Commit.authored_at.desc()).limit(8)
    )
    print(f"commits touching {TARGET}:\n")
    for sha, when, who, msg in rows:
        print(f"  {sha[:8]}  {when.date()}  {who:18.18}  {(msg or '').splitlines()[0][:50]}")

## Top authors (git-history stream)

In [ ]:
with session_scope() as s:
    for who, n in s.execute(select(Commit.author_name, func.count())
                            .group_by(Commit.author_name).order_by(func.count().desc()).limit(8)):
        print(f"  {n:5}  {who}")

## Issues / PRs stream

In [ ]:
with session_scope() as s:
    total = s.scalar(select(func.count()).select_from(Issue))
    if not total:
        print("No issues ingested yet — run `--only-issues` (add GITHUB_TOKEN for the full pull).")
    else:
        for is_pr, state, n in s.execute(select(Issue.is_pull_request, Issue.state, func.count())
                                         .group_by(Issue.is_pull_request, Issue.state)):
            print(f"  {'PR ' if is_pr else 'issue'} [{state:6}]: {n}")
        print("\n  latest:")
        for num, title, is_pr in s.execute(select(Issue.number, Issue.title, Issue.is_pull_request)
                                           .order_by(Issue.created_at.desc()).limit(5)):
            print(f"    #{num:<5} {'PR ' if is_pr else 'iss'} {(title or '')[:55]}")

## Summary

In [ ]:
with session_scope() as s:
    counts = {n: s.scalar(select(func.count()).select_from(m)) for n, m in
              [("files", File), ("commits", Commit), ("commit_files", CommitFile), ("issues", Issue)]}
ok = counts["files"] > 0 and counts["commits"] > 0 and counts["commit_files"] > 0
print("Phase 1 —", "INGESTION OK ✅" if ok else "INCOMPLETE ❌")
print("  ", counts)
if counts["issues"] == 0:
    print("  (issues empty — add GITHUB_TOKEN and run `--only-issues` for the full stream)")